# Phase 1 QLoRA Training

This notebook runs the first QLoRA baseline for the phase-1 structured-output task.

Recommended default:

- model: `Qwen/Qwen2.5-3B-Instruct`
- train file: `data/processed/phase1_sft_train.jsonl`
- val file: `data/processed/phase1_sft_val.jsonl`

In [ ]:
# Run this once in a fresh Jupyter environment if dependencies are missing.
# Comment it out after the environment is stable.

%pip install -q transformers datasets peft accelerate trl bitsandbytes pyyaml jsonschema

In [ ]:
from pathlib import Path
import os


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root from current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
import torch
import platform
import sys

print('python =', sys.version)
print('platform =', platform.platform())
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    print('gpu_count =', torch.cuda.device_count())
    print('bf16_supported =', torch.cuda.is_bf16_supported())

In [ ]:
import yaml
from pathlib import Path

CONFIG_PATH = Path('configs/train/lora_phase1_qwen3b.yaml')
# Fallback if needed:
# CONFIG_PATH = Path('configs/train/lora_phase1_smollm2_1p7b.yaml')

with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

config

In [ ]:
from pathlib import Path
import json

train_file = Path(config['data']['train_file'])
val_file = Path(config['data']['val_file'])

print('train exists =', train_file.exists(), train_file)
print('val exists =', val_file.exists(), val_file)

with train_file.open('r', encoding='utf-8') as handle:
    first_record = json.loads(next(handle))

first_record

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    'json',
    data_files={
        'train': str(train_file),
        'validation': str(val_file),
    },
)

dataset

In [ ]:
from transformers import AutoTokenizer

base_model = config['model']['base_model']
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = 'right'
print('pad_token =', tokenizer.pad_token)
print('eos_token =', tokenizer.eos_token)

In [ ]:
def format_chat_example(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}


dataset = dataset.map(format_chat_example)
dataset['train'][0]['text'][:1000]

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

model.config.use_cache = False
print('model loaded')

In [ ]:
from peft import LoraConfig, TaskType

peft_config = LoraConfig(
    r=config['lora']['r'],
    lora_alpha=config['lora']['alpha'],
    lora_dropout=config['lora']['dropout'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules='all-linear',
)

peft_config

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

output_dir = config['output']['save_dir']

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=float(config['training']['learning_rate']),
    num_train_epochs=int(config['training']['num_train_epochs']),
    per_device_train_batch_size=int(config['training']['per_device_train_batch_size']),
    per_device_eval_batch_size=int(config['training']['per_device_train_batch_size']),
    gradient_accumulation_steps=int(config['training']['gradient_accumulation_steps']),
    warmup_ratio=float(config['training']['warmup_ratio']),
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to='none',
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    peft_config=peft_config,
    args=training_args,
    dataset_text_field='text',
    max_seq_length=int(config['model']['max_seq_length']),
)

trainer

## Pilot Run Suggestion

Before a full run, you can reduce epochs or stop early after confirming the trainer starts correctly.

In [ ]:
train_result = trainer.train()
train_result

In [ ]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print('saved to', output_dir)

## Optional Quick Generation Check

In [ ]:
sample_messages = dataset['validation'][0]['messages']
prompt_text = tokenizer.apply_chat_template(
    sample_messages[:-1],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
generated = model.generate(**inputs, max_new_tokens=256)
decoded = tokenizer.decode(generated[0], skip_special_tokens=True)
print(decoded)